# Travel Agents

Starts the three ADK-backed travel agents: Air Ticketing, Hotel Booking, and Car Rental.
Each connects to the MCP server for tool access (SQL queries against `travel_agency.db`).

In [1]:
import asyncio
import json
import logging
import os
import re
import sys
import uuid
from collections.abc import AsyncIterable
from typing import Any

import uvicorn
import weave
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.genai import types as genai_types

import prompts
from common import MCP_HOST, MCP_PORT, BaseAgent, build_a2a_app

load_dotenv()
logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    stream=sys.stdout,
    force=True,
)
logger.setLevel(logging.INFO)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/Users/paul/Documents/code/a2a-examples/.venv/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


2026-04-16 10:05:19,932 INFO weave.trace.init_message: Logged in as Weights & Biases user: paulbruffett.
View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave
Weave initialized: pbruffett/a2a-lab


In [2]:
class TravelAgent(BaseAgent):
    """ADK-backed travel agent that connects to the MCP server for tools."""

    def __init__(self, agent_name: str, description: str, instructions: str):
        super().__init__(agent_name=agent_name, description=description, content_types=['text', 'text/plain'])
        self.instructions = instructions
        self.agent = None
        self.session_service = InMemorySessionService()
        self.app_name = 'A2A-MCP'
        self.user_id = 'user_1'

    async def _init_agent(self):
        url = f'http://{MCP_HOST}:{MCP_PORT}/sse'
        tools = await MCPToolset(connection_params=SseServerParams(url=url)).get_tools()
        model = os.getenv('LITELLM_MODEL', 'anthropic/claude-haiku-4-5-20251001')
        self.agent = Agent(
            name=self.agent_name, instruction=self.instructions,
            model=LiteLlm(model=model),
            disallow_transfer_to_parent=True, disallow_transfer_to_peers=True,
            generate_content_config=genai_types.GenerateContentConfig(temperature=0.0),
            tools=tools,
        )

    @weave.op()
    async def stream(self, query, context_id, task_id) -> AsyncIterable[dict[str, Any]]:
        logger.info('[%s.stream] REQUEST ctx=%s task=%s query=%r', self.agent_name, context_id, task_id, query)
        if not query:
            raise ValueError('Query cannot be empty')
        if not self.agent:
            await self._init_agent()

        session_id = context_id or uuid.uuid4().hex
        session = await self.session_service.get_session(
            app_name=self.app_name, user_id=self.user_id, session_id=session_id,
        )
        if not session:
            session = await self.session_service.create_session(
                app_name=self.app_name, user_id=self.user_id, session_id=session_id,
            )

        runner = Runner(agent=self.agent, app_name=self.app_name, session_service=self.session_service)
        content = genai_types.Content(role='user', parts=[genai_types.Part(text=query)])
        async for event in runner.run_async(user_id=self.user_id, session_id=session.id, new_message=content):
            if event.is_final_response():
                if event.content and event.content.parts and event.content.parts[0].text:
                    response = '\n'.join(p.text for p in event.content.parts if p.text)
                elif event.content and event.content.parts and any(p.function_response for p in event.content.parts):
                    response = next(p.function_response.model_dump() for p in event.content.parts)
                else:
                    response = f'Error in running agent: {self.agent.name}'
                parsed = self._parse_response(response)
                logger.info('[%s.stream] FINAL %s', self.agent_name, parsed)
                yield parsed
            else:
                yield {'is_task_complete': False, 'require_user_input': False, 'content': f'{self.agent_name}: Processing...'}

    @weave.op()
    def _parse_response(self, raw):
        data = raw
        for pattern in [r'```\n(.*?)\n```', r'```json\s*(.*?)\s*```', r'```tool_outputs\s*(.*?)\s*```']:
            match = re.search(pattern, str(raw), re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group(1))
                    break
                except json.JSONDecodeError:
                    data = match.group(1)
                    break
        if isinstance(data, dict):
            if data.get('status') == 'input_required':
                return {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': data['question']}
            return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': data}
        try:
            data = json.loads(data)
            return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': data}
        except Exception:
            return {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': data}

In [ ]:
agents = [
    (TravelAgent('AirTicketingAgent', 'Book air tickets', prompts.AIRFARE_COT_INSTRUCTIONS), 'agent_cards/air_ticketing_agent.json', 10103),
    (TravelAgent('HotelBookingAgent', 'Book hotels', prompts.HOTELS_COT_INSTRUCTIONS), 'agent_cards/hotel_booking_agent.json', 10104),
    (TravelAgent('CarRentalBookingAgent', 'Book rental cars', prompts.CARS_COT_INSTRUCTIONS), 'agent_cards/car_rental_agent.json', 10105),
]

async def serve_all():
    servers = []
    for agent, card_path, port in agents:
        app = build_a2a_app(agent, card_path)
        config = uvicorn.Config(app=app, host='localhost', port=port, log_level='info',)
        server = uvicorn.Server(config)
        servers.append(server)
        print(f'{agent.agent_name} running at http://localhost:{port}')
    await asyncio.gather(*(s.serve() for s in servers))

await serve_all()

AirTicketingAgent running at http://localhost:10103
HotelBookingAgent running at http://localhost:10104


INFO:     Started server process [8307]
INFO:     Waiting for application startup.
INFO:     Started server process [8307]
INFO:     Waiting for application startup.
INFO:     Started server process [8307]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10103 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://localhost:10104 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://localhost:10105 (Press CTRL+C to quit)


CarRentalBookingAgent running at http://localhost:10105
INFO:     ::1:56744 - "POST / HTTP/1.1" 200 OK
2026-04-16 10:12:30,236 INFO __main__: [AirTicketingAgent.stream] REQUEST ctx=eb0a810d-389d-4aad-b0cb-4a086196ee20 task=7d95de8d-b4bd-4d0b-9040-c8f74268841c query='Book round-trip premium economy class air tickets from San Francisco (SFO) to London (LHR) for 2 travelers for May 1-11, 2026'
2026-04-16 10:12:30,268 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-16 10:12:30,269 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-16 10:12:30,269 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication

/var/folders/ch/wjmp4fss3gvbg92jslrpt8_m0000gn/T/ipykernel_8307/863571684.py:14: DeprecationWarning: MCPToolset class is deprecated, use `McpToolset` instead.
  tools = await MCPToolset(connection_params=SseServerParams(url=url)).get_tools()
/Users/paul/Documents/code/a2a-examples/.venv/lib/python3.13/site-packages/google/adk/tools/mcp_tool/mcp_tool.py:88: UserWarning: [EXPERIMENTAL] BaseAuthenticatedTool: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__(
10:12:30 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic


2026-04-16 10:12:30,282 INFO LiteLLM: 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic


...

2026-04-16 10:13:08,880 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-440f-7253-986e-f7f065a38af8
2026-04-16 10:13:08,882 INFO LiteLLM: 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic
2026-04-16 10:13:09,948 INFO __main__: [HotelBookingAgent.stream] FINAL {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': 'What room type would you prefer? Please choose from: Standard, Single, Double, or Suite?'}
INFO:     ::1:56826 - "POST / HTTP/1.1" 200 OK
2026-04-16 10:13:12,215 INFO __main__: [HotelBookingAgent.stream] REQUEST ctx=005d3c60-bfd6-4554-8ebb-afcd9a6ae05d task=6fd5fd03-f413-46e0-95dd-f5389f96b30d query='Suite'


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-5117-789e-96d7-aefea745d9ff
10:13:12 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic


2026-04-16 10:13:12,217 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9748-5117-789e-96d7-aefea745d9ff
2026-04-16 10:13:12,218 INFO LiteLLM: 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic


10:13:13 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic


2026-04-16 10:13:13,714 INFO LiteLLM: 
LiteLLM completion() model= claude-haiku-4-5-20251001; provider = anthropic
2026-04-16 10:13:17,252 INFO __main__: [HotelBookingAgent.stream] FINAL {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': {'name': 'Luxury Hotel', 'city': 'London', 'hotel_type': 'HOTEL', 'room_type': 'SUITE', 'price_per_night': '$189.46', 'check_in_date': '2026-05-01', 'check_out_date': '2026-05-11', 'number_of_nights': 10, 'check_in_time': '3:00 pm', 'check_out_time': '11:00 am', 'total_rate_usd': '$1,894.60', 'status': 'BOOKING_CONFIRMED', 'description': 'Booking Complete'}}
